In [ ]:
!pip -q install tensorflow-hub scipy pandas

import os, sys, io, json, time, random, shutil, contextlib, platform
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import tensorflow as tf
import cv2

from google.colab import drive
drive.mount("/content/drive")

pass
pass
pass
pass

In [ ]:
CODE_DIR = Path("/content/drive/MyDrive/Fitness")
DATASET_ROOT = CODE_DIR / "kinetics700_subset" / "videos"
OUTPUT_DIR = CODE_DIR / "online_realtime_benchmark_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from movenet import extract_keypoints_from_frame
from assessment import (
    extract_pushup_angles_confidence_aware,
    extract_squat_angles_confidence_aware,
    extract_situp_angles,
    gaussian_smoothing,
    asses_form_of_pushup,
    asses_form_of_squat,
    asses_form_of_situps,
)

pass
pass

In [ ]:
RANDOM_SEED = 42
MAX_VIDEOS_PER_GROUP = None
VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov", ".mkv", ".webm")

CALIBRATION_SECONDS = 3.0
SMOOTHING_BUFFER = 31
MIN_FRAMES_BETWEEN_REPS = 8

DOWN_RATIO = {"push_up":0.35, "squat":0.35, "situp":0.30}
UP_RATIO   = {"push_up":0.75, "squat":0.75, "situp":0.70}
MIN_MOTION_RANGE = {"push_up":15.0, "squat":20.0, "situp":12.0}

GROUPS = {
    ("train","push_up"): DATASET_ROOT/"train"/"push_up",
    ("train","squat"):   DATASET_ROOT/"train"/"squat",
    ("train","situp"):   DATASET_ROOT/"train"/"situp",
    ("val","push_up"):   DATASET_ROOT/"val"/"push_up",
    ("val","squat"):     DATASET_ROOT/"val"/"squat",
    ("val","situp"):     DATASET_ROOT/"val"/"situp",
}

for (split, exercise), folder in GROUPS.items():
    n = 0
    if folder.exists():
        n = len([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS])
    pass

In [ ]:
def find_ground_truth_csv(exercise):
    candidates = {
        "push_up":[CODE_DIR/"pushup_ground_truth.csv", CODE_DIR/"push_up_ground_truth.csv"],
        "squat":[CODE_DIR/"squat_ground_truth.csv"],
        "situp":[CODE_DIR/"situp_ground_truth.csv", CODE_DIR/"sit_up_ground_truth.csv"],
    }
    for p in candidates[exercise]:
        if p.exists():
            return p
    return None

def load_ground_truth(csv_path):
    if csv_path is None:
        return {}
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    name_col = next((c for c in ["video_name","video","filename","file"] if c in df.columns), None)
    gt_col = next((c for c in ["ground_truth","gt","repetitions","count"] if c in df.columns), None)
    if name_col is None or gt_col is None:
        raise ValueError(f"Unrecognized GT columns in {csv_path}: {list(df.columns)}")
    return {str(r[name_col]).strip(): int(r[gt_col]) for _, r in df.iterrows()}

GROUND_TRUTH = {}
for ex in ["push_up","squat","situp"]:
    p = find_ground_truth_csv(ex)
    GROUND_TRUTH[ex] = load_ground_truth(p)
    pass

In [ ]:
@contextlib.contextmanager
def suppress_stdout():
    with contextlib.redirect_stdout(io.StringIO()):
        yield

def sync_tensor(tensor):
    if hasattr(tensor, "numpy"):
        _ = tensor.numpy()

def select_videos(folder, max_videos, seed, ground_truth=None):
    videos = sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS])
    if ground_truth:
        videos = [p for p in videos if p.name in ground_truth]
    if max_videos is None or len(videos) <= max_videos:
        return videos
    rng = random.Random(seed)
    return sorted(rng.sample(videos, max_videos))

def extract_single_angle(kp, exercise):
    seq = [kp]
    if exercise == "push_up":
        vals = extract_pushup_angles_confidence_aware(seq)
    elif exercise == "squat":
        vals = extract_squat_angles_confidence_aware(seq)
    elif exercise == "situp":
        vals, _, _ = extract_situp_angles(seq)
    else:
        raise ValueError(exercise)

    if vals is None or len(vals) == 0:
        return np.nan
    v = float(vals[-1])
    return v if np.isfinite(v) else np.nan

def causal_smoothed_last(raw_buffer):
    if len(raw_buffer) == 0:
        return np.nan
    vals = list(raw_buffer)
    if len(vals) < 3:
        return float(vals[-1])
    try:
        sm = gaussian_smoothing(vals)
        return float(sm[-1]) if len(sm) else float(vals[-1])
    except Exception:
        return float(vals[-1])

In [ ]:
class OnlineRepCounter:
    def __init__(self, exercise, fps):
        self.exercise = exercise
        self.fps = float(fps) if fps and fps > 0 else 30.0
        self.calibration_frames = max(1, int(round(CALIBRATION_SECONDS*self.fps)))
        self.raw_angles = deque(maxlen=SMOOTHING_BUFFER)
        self.calibration_angles = []
        self.calibrated = False
        self.calibration_status = "collecting"
        self.down_th = np.nan
        self.up_th = np.nan
        self.motion_range = np.nan
        self.state = "UP"
        self.rep_bottom = float("inf")
        self.frames_since_last_rep = 999
        self.reps = 0
        self.valid_angle_frames = 0
        self.invalid_angle_frames = 0

    def _state_update(self, a):
        if not np.isfinite(a):
            return False

        self.frames_since_last_rep += 1
        rep_completed = False

        if self.state == "UP":
            if a < self.down_th:
                self.state = "DOWN"
                self.rep_bottom = a

        elif self.state == "DOWN":
            self.rep_bottom = min(self.rep_bottom, a)

            if a > self.up_th and self.frames_since_last_rep >= MIN_FRAMES_BETWEEN_REPS:
                if self.rep_bottom < self.down_th:
                    self.reps += 1
                    rep_completed = True

                self.state = "UP"
                self.rep_bottom = float("inf")
                self.frames_since_last_rep = 0

        return rep_completed

    def _finish_calibration(self):
        valid = np.asarray([a for a in self.calibration_angles if np.isfinite(a)], dtype=float)

        if len(valid) == 0:
            self.calibration_status = "no_valid_angles"
            return False

        mn = float(valid.min())
        mx = float(valid.max())
        self.motion_range = mx - mn

        if self.motion_range < MIN_MOTION_RANGE[self.exercise]:
            self.calibration_status = "insufficient_motion"
            return False

        self.down_th = mn + DOWN_RATIO[self.exercise]*self.motion_range
        self.up_th   = mn + UP_RATIO[self.exercise]*self.motion_range

        self.calibrated = True
        self.calibration_status = "ok"

        
        for a in self.calibration_angles:
            self._state_update(a)

        return True

    def update(self, raw_angle, frame_index):
        if not np.isfinite(raw_angle):
            self.invalid_angle_frames += 1
            return np.nan, False

        self.valid_angle_frames += 1
        self.raw_angles.append(float(raw_angle))
        smoothed = causal_smoothed_last(self.raw_angles)

        if not self.calibrated:
            self.calibration_angles.append(smoothed)
            if frame_index + 1 >= self.calibration_frames:
                self._finish_calibration()
            return smoothed, False

        return smoothed, self._state_update(smoothed)

In [ ]:
OFFLINE_FUNCTIONS = {
    "push_up": asses_form_of_pushup,
    "squat": asses_form_of_squat,
    "situp": asses_form_of_situps,
}

def offline_reference_count(keypoints, exercise):
    with suppress_stdout():
        out = OFFLINE_FUNCTIONS[exercise](keypoints)
    if isinstance(out, (list, tuple)) and len(out) > 0:
        return int(out[0])
    return np.nan

In [ ]:

warmup_video = select_videos(GROUPS[("train","push_up")], 1, RANDOM_SEED)[0]
cap = cv2.VideoCapture(str(warmup_video))
ok, frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Could not decode warm-up frame.")

for _ in range(10):
    kp, _ = extract_keypoints_from_frame(frame)
    sync_tensor(kp)

pass

In [ ]:
def benchmark_streaming_video(video_path, split, exercise):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError("Could not open video")

    source_fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(source_fps) or source_fps <= 0:
        source_fps = 30.0

    reported_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    counter = OnlineRepCounter(exercise, source_fps)

    all_keypoints = []

    decode_ms = []
    movenet_ms = []
    angle_ms = []
    logic_ms = []
    total_frame_ms = []

    frame_idx = 0
    wall0 = time.perf_counter()

    while True:
        frame0 = time.perf_counter()

        t0 = time.perf_counter()
        ok, frame = cap.read()
        t1 = time.perf_counter()

        if not ok:
            break

        decode_ms.append((t1-t0)*1000.0)

        t0 = time.perf_counter()
        kp, _ = extract_keypoints_from_frame(frame)
        sync_tensor(kp)
        t1 = time.perf_counter()
        movenet_ms.append((t1-t0)*1000.0)

        all_keypoints.append(kp)

        t0 = time.perf_counter()
        raw_angle = extract_single_angle(kp, exercise)
        t1 = time.perf_counter()
        angle_ms.append((t1-t0)*1000.0)

        t0 = time.perf_counter()
        _, _ = counter.update(raw_angle, frame_idx)
        t1 = time.perf_counter()
        logic_ms.append((t1-t0)*1000.0)

        total_frame_ms.append((time.perf_counter()-frame0)*1000.0)
        frame_idx += 1

    cap.release()
    wall_s = time.perf_counter() - wall0

    if frame_idx == 0:
        raise RuntimeError("No frames decoded")

    duration_s = frame_idx/source_fps
    effective_fps = frame_idx/wall_s
    rtf = wall_s/duration_s

    arr = np.asarray(total_frame_ms, dtype=float)
    source_deadline = 1000.0/source_fps
    deadline30 = 1000.0/30.0

    offline_reps = offline_reference_count(all_keypoints, exercise)
    gt = GROUND_TRUTH.get(exercise, {}).get(video_path.name, np.nan)

    return {
        "split": split,
        "exercise": exercise,
        "video_name": video_path.name,
        "num_frames": frame_idx,
        "reported_frame_count": reported_frames,
        "source_fps": source_fps,
        "video_duration_s": duration_s,

        "calibration_status": counter.calibration_status,
        "calibrated": counter.calibrated,
        "calibration_motion_range_deg": counter.motion_range,
        "down_threshold_deg": counter.down_th,
        "up_threshold_deg": counter.up_th,

        "online_reps": counter.reps,
        "offline_reps": offline_reps,
        "ground_truth": gt,

        "online_vs_offline_abs_diff":
            abs(counter.reps-offline_reps) if np.isfinite(offline_reps) else np.nan,

        "online_abs_error":
            abs(counter.reps-gt) if np.isfinite(gt) else np.nan,

        "offline_abs_error":
            abs(offline_reps-gt) if np.isfinite(gt) and np.isfinite(offline_reps) else np.nan,

        "total_wall_ms": wall_s*1000.0,
        "effective_processing_fps": effective_fps,
        "real_time_factor": rtf,

        "pct_frames_within_source_deadline":
            float(np.mean(arr <= source_deadline)*100.0),

        "pct_frames_within_30fps_deadline":
            float(np.mean(arr <= deadline30)*100.0),

        "decode_ms_mean": float(np.mean(decode_ms)),
        "decode_ms_p95": float(np.percentile(decode_ms,95)),

        "movenet_ms_mean": float(np.mean(movenet_ms)),
        "movenet_ms_p95": float(np.percentile(movenet_ms,95)),

        "angle_ms_mean": float(np.mean(angle_ms)),
        "angle_ms_p95": float(np.percentile(angle_ms,95)),

        "online_logic_ms_mean": float(np.mean(logic_ms)),
        "online_logic_ms_p95": float(np.percentile(logic_ms,95)),

        "frame_total_ms_mean": float(np.mean(total_frame_ms)),
        "frame_total_ms_median": float(np.median(total_frame_ms)),
        "frame_total_ms_p95": float(np.percentile(total_frame_ms,95)),
        "frame_total_ms_max": float(np.max(total_frame_ms)),

        "status": "ok",
        "error": "",
    }

In [ ]:

for exercise in ["push_up","squat","situp"]:
    folder = GROUPS[("train",exercise)]
    gt = GROUND_TRUTH.get(exercise,{})
    vids = select_videos(folder, 1, RANDOM_SEED, gt if gt else None)
    if not vids:
        continue

    row = benchmark_streaming_video(vids[0], "train", exercise)

    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass
    pass

In [ ]:

selected = []

for i, ((split, exercise), folder) in enumerate(GROUPS.items()):
    if not folder.exists():
        continue

    gt = GROUND_TRUTH.get(exercise,{})
    vids = select_videos(folder, MAX_VIDEOS_PER_GROUP, RANDOM_SEED+i, gt if gt else None)

    pass

    for p in vids:
        selected.append((split,exercise,p))

results = []

for idx, (split,exercise,video_path) in enumerate(selected,1):
    pass

    try:
        row = benchmark_streaming_video(video_path, split, exercise)
        pass
    except Exception as e:
        row = {
            "split":split, "exercise":exercise, "video_name":video_path.name,
            "status":"error", "error":str(e)
        }
        pass

    results.append(row)
    pd.DataFrame(results).to_csv(OUTPUT_DIR/"online_realtime_per_video_partial.csv", index=False)

df = pd.DataFrame(results)
df.to_csv(OUTPUT_DIR/"online_realtime_per_video.csv", index=False)

pass
pass

In [ ]:

ok_df = df[df["status"]=="ok"].copy()
summary_rows = []

def add_summary(scope, scope_value, sub):
    if len(sub)==0:
        return

    summary_rows.append({
        "scope":scope,
        "scope_value":scope_value,
        "n_videos":len(sub),
        "n_calibrated":int(sub["calibrated"].sum()),
        "mean_frame_latency_ms":sub["frame_total_ms_mean"].mean(),
        "median_frame_latency_ms":sub["frame_total_ms_median"].median(),
        "p95_frame_latency_ms":sub["frame_total_ms_p95"].mean(),
        "effective_fps_mean":sub["effective_processing_fps"].mean(),
        "rtf_mean":sub["real_time_factor"].mean(),
        "frames_within_30fps_deadline_pct":
            sub["pct_frames_within_30fps_deadline"].mean(),
        "frames_within_source_deadline_pct":
            sub["pct_frames_within_source_deadline"].mean(),
        "movenet_ms_mean":sub["movenet_ms_mean"].mean(),
        "online_logic_ms_mean":sub["online_logic_ms_mean"].mean(),
        "online_offline_count_diff_mean":
            sub["online_vs_offline_abs_diff"].dropna().mean(),
    })

add_summary("overall","all",ok_df)

for ex, sub in ok_df.groupby("exercise"):
    add_summary("exercise",ex,sub)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR/"online_realtime_summary.csv", index=False)
display(summary_df.round(3))

In [ ]:

gt_df = ok_df[ok_df["ground_truth"].notna()].copy()

def count_metrics(sub, pred_col):
    gt = sub["ground_truth"].to_numpy(float)
    pr = sub[pred_col].to_numpy(float)
    err = pr-gt
    ae = np.abs(err)

    return {
        "n":len(sub),
        "mae":np.mean(ae),
        "rmse":np.sqrt(np.mean(err**2)),
        "exact_pct":np.mean(ae==0)*100,
        "within1_pct":np.mean(ae<=1)*100,
        "bias":np.mean(err),
    }

rows = []

if len(gt_df):
    scopes = [("overall","all",gt_df)]
    scopes += [("exercise",ex,sub) for ex,sub in gt_df.groupby("exercise")]

    for scope,val,sub in scopes:
        on = count_metrics(sub,"online_reps")
        off = count_metrics(sub,"offline_reps")

        rows.append({
            "scope":scope, "scope_value":val, "n":on["n"],
            "online_mae":on["mae"],
            "online_rmse":on["rmse"],
            "online_exact_pct":on["exact_pct"],
            "online_within1_pct":on["within1_pct"],
            "online_bias":on["bias"],
            "offline_mae":off["mae"],
            "offline_rmse":off["rmse"],
            "offline_exact_pct":off["exact_pct"],
            "offline_within1_pct":off["within1_pct"],
            "offline_bias":off["bias"],
        })

accuracy_df = pd.DataFrame(rows)

if len(accuracy_df):
    accuracy_df.to_csv(OUTPUT_DIR/"online_vs_offline_counting_accuracy.csv", index=False)
    display(accuracy_df.round(3))
else:
    pass

In [ ]:

paper_rows = []

for ex, sub in ok_df.groupby("exercise"):
    paper_rows.append({
        "Exercise":ex,
        "N videos":len(sub),
        "Mean latency (ms/frame)":sub["frame_total_ms_mean"].mean(),
        "P95 latency (ms/frame)":sub["frame_total_ms_p95"].mean(),
        "Effective FPS":sub["effective_processing_fps"].mean(),
        "Frames <=33.3 ms (%)":sub["pct_frames_within_30fps_deadline"].mean(),
        "RTF":sub["real_time_factor"].mean(),
        "Online-offline |count diff|":
            sub["online_vs_offline_abs_diff"].dropna().mean(),
    })

if len(ok_df):
    paper_rows.append({
        "Exercise":"Overall",
        "N videos":len(ok_df),
        "Mean latency (ms/frame)":ok_df["frame_total_ms_mean"].mean(),
        "P95 latency (ms/frame)":ok_df["frame_total_ms_p95"].mean(),
        "Effective FPS":ok_df["effective_processing_fps"].mean(),
        "Frames <=33.3 ms (%)":ok_df["pct_frames_within_30fps_deadline"].mean(),
        "RTF":ok_df["real_time_factor"].mean(),
        "Online-offline |count diff|":
            ok_df["online_vs_offline_abs_diff"].dropna().mean(),
    })

paper_df = pd.DataFrame(paper_rows)
paper_df.to_csv(OUTPUT_DIR/"paper_ready_online_realtime_table.csv", index=False)
display(paper_df.round(3))

In [ ]:

gpu_info = []
for dev in tf.config.list_physical_devices("GPU"):
    try:
        gpu_info.append({
            "device":str(dev),
            "details":tf.config.experimental.get_device_details(dev),
        })
    except Exception:
        gpu_info.append({"device":str(dev)})

metadata = {
    "dataset_root":str(DATASET_ROOT),
    "calibration_seconds":CALIBRATION_SECONDS,
    "smoothing_buffer_frames":SMOOTHING_BUFFER,
    "down_ratio":DOWN_RATIO,
    "up_ratio":UP_RATIO,
    "minimum_motion_range_deg":MIN_MOTION_RANGE,
    "minimum_frames_between_reps":MIN_FRAMES_BETWEEN_REPS,
    "timing_scope":{
        "decode":"OpenCV VideoCapture.read per frame",
        "movenet":"extract_keypoints_from_frame with TensorFlow synchronization",
        "angle":"exercise-specific angle extraction on current frame",
        "logic":"causal smoothing + calibration + finite-state update",
        "frame_total":"decode + MoveNet + angle + online logic",
    },
    "software":{
        "python":platform.python_version(),
        "tensorflow":tf.__version__,
        "opencv":cv2.__version__,
        "numpy":np.__version__,
        "pandas":pd.__version__,
    },
    "gpu":gpu_info,
}

(OUTPUT_DIR/"online_realtime_metadata.json").write_text(
    json.dumps(metadata, indent=2, default=str), encoding="utf-8"
)

bundle_dir = OUTPUT_DIR/"online_realtime_bundle"
bundle_dir.mkdir(exist_ok=True)

for name in [
    "online_realtime_per_video.csv",
    "online_realtime_summary.csv",
    "online_vs_offline_counting_accuracy.csv",
    "paper_ready_online_realtime_table.csv",
    "online_realtime_metadata.json",
]:
    src = OUTPUT_DIR/name
    if src.exists():
        shutil.copy2(src, bundle_dir/name)

zip_path = shutil.make_archive(
    str(OUTPUT_DIR/"online_realtime_bundle"),
    "zip",
    root_dir=bundle_dir
)

pass